# Lab3 - Multimodal Feature Builder (Videos + Whisper text + Sensorial)

This notebook is prepared to run on the cluster with the same environment and hardware assumptions used in `run.sh` and `run_parallel.sh`.

Pipeline goals:
1. Load Whisper transcripts and sensorial data from EXIST 2026 Videos Dataset.
2. Extract text embeddings with XLM-RoBERTa.
3. Extract keyframes with PySceneDetect.
4. Build early-fusion vectors by concatenating text + video + sensorial features.
5. Save artifacts for the three training configurations (ES, EN, ES+EN).


In [1]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.impute import SimpleImputer

from transformers import AutoTokenizer, AutoModel

from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector

import cv2


In [ ]:
# Cluster configuration copied from run.sh / run_parallel.sh
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_SEQUENTIAL = 'shard:6'  # run.sh
SLURM_GRES_PARALLEL = 'shard:4'    # run_parallel.sh
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

CLUSTER_DATASET_ROOT = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset')
CLUSTER_TRAINING_ROOT = CLUSTER_DATASET_ROOT / 'training'

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    # Fallback for execution from notebooks_shiyi directory
    if (cwd / 'artifacts').exists() or (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('Could not resolve project root with materials/dataset_task3_exist2026.')

def resolve_train_json(project_root: Path) -> Path:
    local_candidate = project_root.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json'
    cluster_candidate = CLUSTER_TRAINING_ROOT / 'EXIST2026_training.json'
    if local_candidate.exists():
        return local_candidate
    if cluster_candidate.exists():
        return cluster_candidate
    raise FileNotFoundError('Training JSON not found in local materials or cluster path.')

def resolve_test_json(project_root: Path, train_json_path: Path) -> Path:
    local_candidate = project_root.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'
    cluster_candidate = train_json_path.parent / 'test.json'
    if local_candidate.exists():
        return local_candidate
    if cluster_candidate.exists():
        return cluster_candidate
    raise FileNotFoundError('Test JSON not found in local materials or cluster path.')

PROJECT_ROOT = resolve_project_root()
TRAIN_JSON_PATH = resolve_train_json(PROJECT_ROOT)
TEST_JSON_PATH = resolve_test_json(PROJECT_ROOT, TRAIN_JSON_PATH)
VIDEO_ROOT = TRAIN_JSON_PATH.parent

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
KEYFRAME_DIR = PROJECT_ROOT / 'keyframes'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
KEYFRAME_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_SEARCH_ROOTS = []
for root in [VIDEO_ROOT, VIDEO_ROOT.parent, TEST_JSON_PATH.parent, CLUSTER_TRAINING_ROOT, CLUSTER_DATASET_ROOT]:
    if root.exists() and root not in VIDEO_SEARCH_ROOTS:
        VIDEO_SEARCH_ROOTS.append(root)

def resolve_video_abs_path(path_video: str) -> str:
    pv = Path(str(path_video))
    if pv.is_absolute() and pv.exists():
        return str(pv.resolve())

    candidates = []
    for base in VIDEO_SEARCH_ROOTS:
        candidates.append((base / pv).resolve())
        if not str(pv).startswith('training/') and not str(base).endswith('/training'):
            candidates.append((base / 'training' / pv).resolve())

    for c in candidates:
        if c.exists():
            return str(c)

    # Keep a deterministic fallback path for debug purposes.
    if candidates:
        return str(candidates[0])
    return str(pv.resolve())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_JSON_PATH:', TRAIN_JSON_PATH)
print('TEST_JSON_PATH:', TEST_JSON_PATH)
print('VIDEO_SEARCH_ROOTS:', [str(p) for p in VIDEO_SEARCH_ROOTS])
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', 'not set'))
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])

Device: cuda
PROJECT_ROOT: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi
TRAIN_JSON_PATH: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/materials/dataset_task3_exist2026/training.json
VIDEO_SEARCH_ROOTS: ['/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/materials/dataset_task3_exist2026', '/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/materials', '/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training', '/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset']
CUDA_VISIBLE_DEVICES: not set
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [ ]:
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)

def build_rows(raw_obj, source_tag):
    rows = []
    for sample_id, payload in raw_obj.items():
        rec = {'sample_id': str(sample_id), 'source': source_tag}
        rec.update(payload)
        rows.append(rec)
    return rows

df_train = pd.DataFrame(build_rows(train_raw, 'train'))
df_test = pd.DataFrame(build_rows(test_raw, 'test'))

def majority_task3_1(label_list):
    vals = [x for x in label_list if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df_train['label_task3_1_majority'] = df_train['labels_task3_1'].apply(majority_task3_1)
df_train['y'] = df_train['label_task3_1_majority'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)

df_test['label_task3_1_majority'] = 'UNKNOWN'
df_test['y'] = -1

df = pd.concat([df_train, df_test], ignore_index=True)
df = df.drop_duplicates(subset=['sample_id'], keep='last').sort_values('sample_id').reset_index(drop=True)
df['source'] = df['source'].astype(str)
df['lang'] = df['lang'].fillna('').astype(str).str.lower()

df['video_abs_path'] = df['path_video'].apply(resolve_video_abs_path)
df['video_exists'] = df['video_abs_path'].apply(lambda p: Path(p).exists())

missing_video_count = int((~df['video_exists']).sum())

print('Total samples:', len(df), '| train:', int((df['source'] == 'train').sum()), '| test:', int((df['source'] == 'test').sum()))
print('Languages:', df['lang'].value_counts().to_dict())
print('Label distribution (train):', df.loc[df['source'] == 'train', 'label_task3_1_majority'].value_counts().to_dict())
print('Missing videos after path resolution:', missing_video_count)

if missing_video_count > 0:
    print('Missing video examples:')
    print(df.loc[~df['video_exists'], ['sample_id', 'path_video', 'video_abs_path']].head(5).to_string(index=False))

df[['sample_id', 'source', 'lang', 'split', 'text', 'path_video', 'video_exists', 'y']].head(3)


Total samples: 2006
Languages: {'es': 1212, 'en': 794}
Label distribution: {'NO': 1045, 'YES': 961}
Missing videos after path resolution: 0


,sample_id,lang,split,text,path_video,video_exists,y
0,120001,es,TRAIN-VIDEO_ES,cuando ves a las del 08 en la fiesta tu amigo...,videos/7281385962049998086.mp4,True,1
1,120002,es,TRAIN-VIDEO_ES,mujer sola caminando por la calle | yo automat...,videos/7164058026352168197.mp4,True,1
2,120003,es,TRAIN-VIDEO_ES,mi amigo no le importa ni las mujeres ni las ...,videos/7248606026386263323.mp4,True,0


In [5]:
# Flatten nested sensorial metrics by averaging values across users per modality.
def flatten_sensorial(sensorial_obj):
    if not isinstance(sensorial_obj, dict):
        return {}

    result = {}
    modalities = sensorial_obj.get('modalities', {})
    for mod_name, mod_payload in modalities.items():
        by_user = mod_payload.get('by_user', {}) if isinstance(mod_payload, dict) else {}
        agg = {}
        count = {}

        for _, feat_map in by_user.items():
            if not isinstance(feat_map, dict):
                continue
            for feat_name, feat_val in feat_map.items():
                if feat_val is None:
                    continue
                if isinstance(feat_val, (int, float, np.integer, np.floating)):
                    key = f'{mod_name.lower()}__{feat_name}'
                    agg[key] = agg.get(key, 0.0) + float(feat_val)
                    count[key] = count.get(key, 0) + 1

        for k, v in agg.items():
            result[k] = v / max(count[k], 1)

    return result


sensorial_series = df['sensorial'].apply(flatten_sensorial)
sensorial_df = pd.json_normalize(sensorial_series).astype(float)

if sensorial_df.shape[1] == 0:
    raise RuntimeError('No sensorial features were extracted.')

imputer = SimpleImputer(strategy='constant', fill_value=0.0)
sensorial_matrix = imputer.fit_transform(sensorial_df)
sensorial_columns = sensorial_df.columns.tolist()

print('Sensorial matrix shape:', sensorial_matrix.shape)
print('Sensorial feature count:', len(sensorial_columns))


Sensorial matrix shape: (2006, 108)
Sensorial feature count: 108


In [6]:
# XLM-R text embeddings from Whisper transcripts in field `text`.
TEXT_MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 256
BATCH_SIZE = 16

texts = df['text'].fillna('').astype(str).tolist()

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_model.eval()

all_embeds = []

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='Text embeddings'):
    batch_texts = texts[start:start + BATCH_SIZE]
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        out = text_model(**enc)
        cls_emb = out.last_hidden_state[:, 0, :]

    all_embeds.append(cls_emb.cpu().numpy())

text_matrix = np.vstack(all_embeds).astype(np.float32)
print('Text matrix shape:', text_matrix.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text embeddings:   0%|          | 0/126 [00:00<?, ?it/s]

Text matrix shape: (2006, 768)


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from scenedetect import detect, ContentDetector

# Reduce noisy ffmpeg/opencv logs from partially corrupt videos.
os.environ['OPENCV_FFMPEG_CAPTURE_OPTIONS'] = 'loglevel;error'
if hasattr(cv2, 'setLogLevel'):
    try:
        cv2.setLogLevel(0)
    except Exception:
        pass

def extract_keyframes(video_path, output_dir, threshold=30.0, max_keyframes=6):
    output_dir.mkdir(parents=True, exist_ok=True)
    video_str = str(video_path)

    if not Path(video_path).exists():
        return [], 'missing_file'

    scenes = []
    scene_status = 'ok'
    try:
        scenes = detect(video_str, ContentDetector(threshold=threshold))
    except Exception:
        scenes = []
        scene_status = 'scene_detect_error'

    cap = cv2.VideoCapture(video_str)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return [], 'unreadable_video'

    frame_indices = []
    if len(scenes) == 0:
        frame_indices = [frame_count // 2]
    else:
        for start_tc, end_tc in scenes:
            s = start_tc.get_frames()
            e = end_tc.get_frames()
            frame_indices.append((s + e) // 2)

    frame_indices = frame_indices[:max_keyframes]
    keyframe_paths = []

    for idx_num, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        frame_path = output_dir / f'kf_{idx_num:02d}_{int(frame_idx)}.jpg'
        cv2.imwrite(str(frame_path), frame)
        keyframe_paths.append(frame_path)

    cap.release()

    if not keyframe_paths:
        if scene_status != 'ok':
            return [], scene_status
        return [], 'decode_error'

    if scene_status != 'ok':
        return keyframe_paths, scene_status
    return keyframe_paths, 'ok'

def keyframe_stats(image_paths):
    if not image_paths:
        return {
            'video__kf_count': 0.0,
            'video__r_mean': 0.0,
            'video__g_mean': 0.0,
            'video__b_mean': 0.0,
            'video__r_std': 0.0,
            'video__g_std': 0.0,
            'video__b_std': 0.0,
            'video__sharpness': 0.0,
        }

    means = []
    stds = []
    sharp = []

    for p in image_paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        means.append(rgb.reshape(-1, 3).mean(axis=0))
        stds.append(rgb.reshape(-1, 3).std(axis=0))
        sharp.append(float(cv2.Laplacian(rgb, cv2.CV_64F).var()))

    if not means:
        return {
            'video__kf_count': 0.0,
            'video__r_mean': 0.0,
            'video__g_mean': 0.0,
            'video__b_mean': 0.0,
            'video__r_std': 0.0,
            'video__g_std': 0.0,
            'video__b_std': 0.0,
            'video__sharpness': 0.0,
        }

    means = np.array(means)
    stds = np.array(stds)

    return {
        'video__kf_count': float(len(means)),
        'video__r_mean': float(means[:, 0].mean()),
        'video__g_mean': float(means[:, 1].mean()),
        'video__b_mean': float(means[:, 2].mean()),
        'video__r_std': float(stds[:, 0].mean()),
        'video__g_std': float(stds[:, 1].mean()),
        'video__b_std': float(stds[:, 2].mean()),
        'video__sharpness': float(np.mean(sharp)),
    }

video_feats = []
video_status_rows = []
for row in tqdm(df.itertuples(index=False), total=len(df), desc='Keyframes + video stats'):
    sample_id = str(row.sample_id)
    sample_dir = KEYFRAME_DIR / sample_id
    video_path = Path(row.video_abs_path)

    kfs, status = extract_keyframes(video_path, sample_dir)
    stats = keyframe_stats(kfs)
    stats['sample_id'] = sample_id
    video_feats.append(stats)
    video_status_rows.append({
        'sample_id': sample_id,
        'video_path': str(video_path),
        'status': status,
        'keyframes_extracted': len(kfs),
    })

video_df = pd.DataFrame(video_feats).sort_values('sample_id').reset_index(drop=True)
video_feature_cols = [c for c in video_df.columns if c != 'sample_id']
video_matrix = video_df[video_feature_cols].to_numpy(dtype=np.float32)

video_status_df = pd.DataFrame(video_status_rows)
status_counts = video_status_df['status'].value_counts().to_dict()
video_status_df.to_csv(ARTIFACTS_DIR / 'video_extraction_status.csv', index=False)

print('Video matrix shape:', video_matrix.shape)
print('Video extraction status counts:', status_counts)
print('Saved status file:', ARTIFACTS_DIR / 'video_extraction_status.csv')

Keyframes + video stats:   0%|          | 0/2006 [00:00<?, ?it/s]

[h264 @ 0x3ecbfd40] Invalid NAL unit size (4049 > 274).
[h264 @ 0x3ecbfd40] missing picture in access unit with size 278
[h264 @ 0x6968b940] Invalid NAL unit size (4049 > 274).
[h264 @ 0x6968b940] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 0, offset 0x100ebf: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 1, offset 0x10136d: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 0, offset 0x101426: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 1, offset 0x1016ae: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 0, offset 0x101789: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 0, offset 0x101af4: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 1, offset 0x103089: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7dfaec049fc0] stream 0, offset 0x103141: partial file
[h264 @ 0x40fed540] Invalid NAL unit size (4049 > 274).
[h264 @ 0x40fed540] missing picture in ac

In [ ]:
meta_df = df[['sample_id', 'source', 'lang', 'split', 'y', 'label_task3_1_majority']].copy()
meta_df = meta_df.sort_values('sample_id').reset_index(drop=True)

sensorial_out_df = pd.DataFrame(sensorial_matrix, columns=sensorial_columns)
sensorial_out_df['sample_id'] = df['sample_id'].values
sensorial_out_df = sensorial_out_df.sort_values('sample_id').reset_index(drop=True)

text_out_df = pd.DataFrame(text_matrix)
text_out_df['sample_id'] = df['sample_id'].values
text_out_df = text_out_df.sort_values('sample_id').reset_index(drop=True)

video_df = video_df.sort_values('sample_id').reset_index(drop=True)

assert list(meta_df['sample_id']) == list(sensorial_out_df['sample_id']) == list(text_out_df['sample_id']) == list(video_df['sample_id'])

X_text = text_out_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)
X_video = video_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)
X_sensor = sensorial_out_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)

X_fusion = np.hstack([X_text, X_video, X_sensor]).astype(np.float32)
y = meta_df['y'].to_numpy(dtype=np.int64)
langs = meta_df['lang'].astype(str).to_numpy()
sample_ids = meta_df['sample_id'].astype(str).to_numpy()
sources = meta_df['source'].astype(str).to_numpy()

train_mask = sources == 'train'
test_mask = sources == 'test'
if not train_mask.any() or not test_mask.any():
    raise RuntimeError('Expected both train and test rows in combined dataframe.')

X_fusion_train = X_fusion[train_mask]
y_train = y[train_mask]
langs_train = langs[train_mask]
sample_ids_train = sample_ids[train_mask]

X_fusion_test = X_fusion[test_mask]
y_test = np.full(int(test_mask.sum()), -1, dtype=np.int64)
langs_test = langs[test_mask]
sample_ids_test = sample_ids[test_mask]

print('Early fusion shape:', X_fusion.shape)
print('Text dim:', X_text.shape[1], '| Video dim:', X_video.shape[1], '| Sensor dim:', X_sensor.shape[1])
print('Train rows:', int(train_mask.sum()), '| Test rows:', int(test_mask.sum()))

# Backward-compatible train artifact.
np.savez_compressed(
    ARTIFACTS_DIR / 'fusion_features.npz',
    X_fusion=X_fusion_train,
    y=y_train,
    langs=langs_train,
    sample_ids=sample_ids_train,
    text_dim=np.array([X_text.shape[1]], dtype=np.int64),
    video_dim=np.array([X_video.shape[1]], dtype=np.int64),
    sensor_dim=np.array([X_sensor.shape[1]], dtype=np.int64),
)

# Split artifacts for separate train/inference pipelines.
np.savez_compressed(
    ARTIFACTS_DIR / 'fusion_features_train.npz',
    X_fusion=X_fusion_train,
    y=y_train,
    langs=langs_train,
    sample_ids=sample_ids_train,
    text_dim=np.array([X_text.shape[1]], dtype=np.int64),
    video_dim=np.array([X_video.shape[1]], dtype=np.int64),
    sensor_dim=np.array([X_sensor.shape[1]], dtype=np.int64),
)

np.savez_compressed(
    ARTIFACTS_DIR / 'fusion_features_test.npz',
    X_fusion=X_fusion_test,
    y=y_test,
    langs=langs_test,
    sample_ids=sample_ids_test,
    text_dim=np.array([X_text.shape[1]], dtype=np.int64),
    video_dim=np.array([X_video.shape[1]], dtype=np.int64),
    sensor_dim=np.array([X_sensor.shape[1]], dtype=np.int64),
)

meta_train_df = meta_df.loc[train_mask].drop(columns=['source']).reset_index(drop=True)
meta_test_df = meta_df.loc[test_mask].drop(columns=['source']).reset_index(drop=True)
video_train_df = video_df.loc[train_mask].reset_index(drop=True)
video_test_df = video_df.loc[test_mask].reset_index(drop=True)
sensorial_train_df = sensorial_out_df.loc[train_mask].reset_index(drop=True)
sensorial_test_df = sensorial_out_df.loc[test_mask].reset_index(drop=True)

# Backward-compatible train csv artifacts.
meta_train_df.to_csv(ARTIFACTS_DIR / 'sample_metadata.csv', index=False)
video_train_df.to_csv(ARTIFACTS_DIR / 'video_features.csv', index=False)
sensorial_train_df.to_csv(ARTIFACTS_DIR / 'sensorial_features.csv', index=False)

# Explicit split csv artifacts.
meta_train_df.to_csv(ARTIFACTS_DIR / 'sample_metadata_train.csv', index=False)
meta_test_df.to_csv(ARTIFACTS_DIR / 'sample_metadata_test.csv', index=False)
video_train_df.to_csv(ARTIFACTS_DIR / 'video_features_train.csv', index=False)
video_test_df.to_csv(ARTIFACTS_DIR / 'video_features_test.csv', index=False)
sensorial_train_df.to_csv(ARTIFACTS_DIR / 'sensorial_features_train.csv', index=False)
sensorial_test_df.to_csv(ARTIFACTS_DIR / 'sensorial_features_test.csv', index=False)

print('Saved artifacts in:', ARTIFACTS_DIR)
print('Saved split artifacts: fusion_features_train.npz, fusion_features_test.npz')
print('Saved split csv files for metadata/video/sensorial.')
print('Train rows:', len(meta_train_df), '| Test rows:', len(meta_test_df))


NameError: name 'video_df' is not defined

In [ ]:
# Artifact audit: this clarifies why artifacts may look sparse before downstream notebooks run.
from pathlib import Path
import zipfile

artifact_files = sorted([p for p in ARTIFACTS_DIR.glob('*') if p.is_file()])
audio_files = list((ARTIFACTS_DIR / 'audio_wav').glob('*.wav')) if (ARTIFACTS_DIR / 'audio_wav').exists() else []

print('Artifacts directory:', ARTIFACTS_DIR)
print('Artifact files:')
for p in artifact_files:
    print(' -', p.name, '|', p.stat().st_size, 'bytes')

print('audio_wav count:', len(audio_files))
print('sample_metadata rows (incl header):', sum(1 for _ in open(ARTIFACTS_DIR / 'sample_metadata.csv', 'r', encoding='utf-8')))

# Show NPZ entries to verify full content exists.
for npz_name in ['fusion_features.npz', 'sensorial_filtered_features.npz']:
    npz_path = ARTIFACTS_DIR / npz_name
    if npz_path.exists():
        with zipfile.ZipFile(npz_path, 'r') as zf:
            print(f'[{npz_name}] entries:', zf.namelist())

print('NOTE: audio_wav is generated by notebook 05, so it can be sparse after only running notebook 01.')

## Next step

Run one of the training notebooks:
- `02_train_classifier_es.ipynb`
- `03_train_classifier_en.ipynb`
- `04_train_classifier_es_en.ipynb`
